# IRIS — Neural IDS for AI Agent Pipelines

**One-click launcher** for the IRIS interactive dashboard.

This notebook clones the project from GitHub, installs dependencies, downloads pre-trained checkpoints, and launches the Gradio web app with a **public URL** you can open in any browser.

### What you need

**For the full defended-agent demo** (SAE detector + four-layer defense stack + Phi-3.5 agent generating responses):
- A Google Colab runtime with an **L4**, **G4** (RTX PRO 6000), **A100**, or **H100** GPU. Colab Pro or Pro+ is required for these tiers.

**For detection-only mode** (SAE detector + defense layers 2-4, no agent responses):
- A free-tier **T4** runtime works. The notebook will auto-detect that T4 cannot safely load Phi-3.5 (Colab T4 is paired with only 12.7 GB of system RAM, and the Phi-3.5 fp16 weights stream through CPU at ~8 GB before 4-bit quantization) and will skip the agent load to prevent an OOM.

### Instructions
Select `Runtime > Change runtime type`, pick **L4 / G4 / A100** for full agent mode or **T4** for detection-only, then click `Runtime > Run all`.

*Nathan Cheung | York University | CSSD 2221 | Winter 2026*

In [ ]:
# === Mount Google Drive (for caching) and clone repository ===
from google.colab import drive
drive.mount('/content/drive')

# Always fetch latest code from GitHub
import shutil, os
os.chdir('/content')  # ensure valid working directory before cleanup
if os.path.exists('/content/iris'):
    shutil.rmtree('/content/iris')
!git clone https://github.com/ncheung13579/iris.git /content/iris
%cd /content/iris

# transformer-lens pins numpy<2 but works fine with numpy 2.x (pure Python).
# Install it without deps to avoid downgrading Colab's numpy, which would
# require a runtime restart. Its real deps are in requirements.txt.
!pip install transformer-lens --no-deps -q
!grep -v 'transformer-lens' requirements.txt | pip install -r /dev/stdin -q
print('Setup complete.')

In [ ]:
# === Ensure pre-trained checkpoints are present (with Drive cache) ===
# First run: downloads from GitHub Release and caches to Drive.
# Subsequent runs: restores from Drive instantly, even after runtime reset.
# A minimum-size check handles the case where `git clone` produced LFS
# pointer files (a few hundred bytes) instead of the real binaries.
import os, shutil, urllib.request

RELEASE_URL = "https://github.com/ncheung13579/iris/releases/download/v1.0.0"
DRIVE_CACHE = "/content/drive/MyDrive/iris_checkpoints"

# (local_path: (release_asset_name, min_real_size_bytes))
files = {
    "checkpoints/sae_d10240_lambda1e-04.pt":   ("sae_d10240_lambda1e-04.pt", 100_000_000),
    "checkpoints/feature_matrix.npy":          ("feature_matrix.npy",         10_000_000),
    "checkpoints/sensitivity_scores.npy":      ("sensitivity_scores.npy",     10_000),
    "data/processed/iris_dataset_balanced.json": ("iris_dataset_balanced.json", 100_000),
}

os.makedirs(DRIVE_CACHE, exist_ok=True)

def is_real(path: str, min_size: int) -> bool:
    return os.path.exists(path) and os.path.getsize(path) >= min_size

for local_path, (remote_name, min_size) in files.items():
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    drive_path = os.path.join(DRIVE_CACHE, remote_name)

    if is_real(local_path, min_size):
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"  [OK] {local_path} (already present, {size_mb:.1f} MB)")
    elif is_real(drive_path, min_size):
        shutil.copy2(drive_path, local_path)
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"  [OK] {local_path} (restored from Drive, {size_mb:.1f} MB)")
    else:
        # Remove any stale LFS pointer before downloading the real binary.
        if os.path.exists(local_path):
            os.remove(local_path)
        url = f"{RELEASE_URL}/{remote_name}"
        print(f"  Downloading {remote_name}...", end=" ", flush=True)
        urllib.request.urlretrieve(url, local_path)
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        try:
            shutil.copy2(local_path, drive_path)
            print(f"OK ({size_mb:.1f} MB, cached to Drive)")
        except OSError:
            print(f"OK ({size_mb:.1f} MB, Drive cache skipped)")

print("\nAll checkpoints ready.")

In [ ]:
# === Launch the IRIS Detection Dashboard ===
# Loads GPT-2 Large, the SAE, all detectors, and (if VRAM permits)
# the Phi-3.5-mini agent for the full defended-agent demo.
#
# On Colab T4 runtimes, the pipeline auto-detects that the 12.7 GB
# system RAM is too tight to safely load Phi-3.5 and comes up in
# detection-only mode. The SAE detector, feature inspection, and
# defense layers 2-4 all remain fully functional. For the full agent
# demo (with Phi-3.5 responses), use an L4 or A100 runtime.
from src.app import IRISPipeline, build_app

pipeline = IRISPipeline('.')
pipeline.load()
app = build_app(pipeline)

# Suppress Colab's automatic inline rendering by temporarily
# blocking IPython display calls during launch
import IPython.display as _ipd
_original_display = _ipd.display
_ipd.display = lambda *a, **kw: None

try:
    _, local_url, share_url = app.launch(
        share=True,
        prevent_thread_lock=True,
    )
finally:
    _ipd.display = _original_display

url = share_url or local_url

# Show a clickable link + auto-open via JavaScript
_ipd.display(_ipd.HTML(
    f'<script>window.open("{url}", "_blank");</script>'
    f'<div style="padding:16px; background:#f0f9ff; border:2px solid #2563EB; '
    f'border-radius:10px; margin:10px 0; font-size:16px; text-align:center;">'
    f'<b>IRIS Dashboard is live!</b><br><br>'
    f'If it didn\'t open automatically, '
    f'<a href="{url}" target="_blank" style="font-size:18px; color:#2563EB;">'
    f'click here to open IRIS</a><br><br>'
    f'<code style="font-size:13px; opacity:0.7;">{url}</code></div>'
))